# 示例 06 · 多 Agent：子 Agent（Subagents）

**来源：** [LangChain 官方文档 — Multi-Agent Subagents](https://docs.langchain.com/oss/python/langchain/multi-agent/subagents)

---

## 学习目标

完成本 Notebook 后，你将能够：

1. 解释 **监督者 + 子 Agent** 架构，以及其适用场景
2. 实现 **每个 Agent 一个工具（Tool-Per-Agent）** 模式
3. 实现 **单一调度工具（Single Dispatch Tool）** 模式，用一个 `task()` 工具路由到任意 Agent
4. 使用 `Enum` 约束实现 **类型安全的 Agent 发现**
5. 控制流入和流出子 Agent 的 **上下文数据**

---

## 背景介绍

### 单个大 Agent 的问题

当任务变得复杂，单个 Agent 的上下文窗口会被所有子任务的历史记录填满——
这既降低速度，也会混淆推理过程。解决方案是**分解**：
将工作分配给各自拥有独立上下文的专业 Agent。

### 监督者架构

```
        用户请求
           │
           ▼
  ┌─────────────────┐
  │    监督者        │  ← 主 Agent，负责协调工作
  │  (orchestrator) │
  └────────┬────────┘
           │  将子 Agent 作为工具调用
  ┌────────┼────────┐
  ▼        ▼        ▼
调研     写作     审核      ← 每个都是完整的 Agent
 Agent   Agent   Agent
```

核心特性：
- **集中路由** — 所有决策都经过监督者
- **上下文隔离** — 每次子 Agent 调用都从空白上下文开始
- **可组合** — 子 Agent 本身也可以拥有子子 Agent

### 两种调用模式

| 模式 | 优点 | 缺点 |
|------|------|------|
| **每个 Agent 一个工具** | 简单，每个 Agent 可单独定制 | Agent 多时难以维护 |
| **单一调度工具** | 一个工具，N 个 Agent，易于扩展 | 对每个 Agent 的定制化较少 |

请从上到下依次运行每个单元格。

## 环境配置

In [1]:
import sys
from pathlib import Path
from enum import Enum
from typing import Annotated

# 将项目根目录加入 sys.path，使 common/ 包可以被导入
_root = Path().resolve().parent.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from langchain.agents import create_agent
from langchain.tools import tool, InjectedToolCallId
from langchain_core.messages import ToolMessage
from langgraph.types import Command

from common.env import get_env   # 加载 .env 文件
from common.llm import create_llm
from common.tracing import create_langfuse_handler, build_langfuse_config, get_langfuse_host

handler = create_langfuse_handler()

def make_config(trace_name: str, thread_id: str = "s06-cn") -> dict:
    """构建包含 Langfuse 追踪信息的配置字典。"""
    cfg = build_langfuse_config(handler, session_id="s06-subagents-cn", trace_name=trace_name)
    cfg["configurable"] = {"thread_id": thread_id}
    return cfg

print("✓ 环境配置完成")

✓ 环境配置完成


---

## 第一部分 · 每个 Agent 一个工具（Tool-Per-Agent）

最简单的模式：分别创建每个子 Agent，然后用 `@tool` 装饰器将其包装成工具，
使监督者可以像调用普通工具一样调用它。

```
监督者
  ├─ call_research_agent(query)  ←── @tool 包装 research_agent.invoke()
  └─ call_writer_agent(brief)   ←── @tool 包装 writer_agent.invoke()
```

### 步骤 1a — 创建带有专属工具的子 Agent

In [2]:
# ── 调研子 Agent 的工具 ────────────────────────────────────────────────────────
@tool
def fetch_facts(topic: str) -> str:
    """从知识库中获取某个主题的关键事实。"""
    # 模拟知识库查询（实际项目中替换为真实 API 或 RAG 检索）
    facts = {
        "python": "Python 由 Guido van Rossum 于 1991 年创建，强调代码可读性。",
        "langchain": "LangChain 是一个开源框架，用于构建基于大语言模型的应用程序。",
        "agents": "AI Agent 将大语言模型与工具结合，能够自主执行多步骤任务。",
    }
    key = next((k for k in facts if k in topic.lower()), None)
    return facts[key] if key else f"没有关于 '{topic}' 的缓存事实，将基于训练数据推理。"

# ── 写作子 Agent 的工具 ────────────────────────────────────────────────────────
@tool
def count_words(text: str) -> str:
    """统计草稿中的字数，并检查是否满足 50 词的最低要求。"""
    n = len(text.split())
    status = "✓ 满足最低要求" if n >= 50 else f"✗ 还需要 {50 - n} 个词"
    return f"{n} 个词 — {status}"

# ── 创建子 Agent ──────────────────────────────────────────────────────────────
research_agent = create_agent(
    model=create_llm(),
    tools=[fetch_facts],
    system_prompt=(
        "你是一名调研专家。使用 fetch_facts 工具收集信息，"
        "然后简洁地总结你的发现。请用中文回复。"
    ),
)

writer_agent = create_agent(
    model=create_llm(),
    tools=[count_words],
    system_prompt=(
        "你是一名写作专家。根据调研结果，"
        "撰写清晰易懂的说明文字。使用 count_words 验证字数。请用中文写作。"
    ),
)

print("✓ 子 Agent 创建完成")

✓ 子 Agent 创建完成


### 步骤 1b — 将子 Agent 包装为工具

包装函数的职责：
1. 将输入格式化为子 Agent 期望的格式
2. 调用 `subagent.invoke()`
3. 只提取并返回最终消息内容——让监督者的上下文保持简洁

In [3]:
@tool("research", description="调研某个主题并返回事实性摘要。")
def call_research_agent(query: str) -> str:
    """将调研任务委托给调研子 Agent。"""
    result = research_agent.invoke(
        {"messages": [{"role": "user", "content": query}]},
        config=make_config("调研子Agent", "s06-cn-research"),
    )
    # create_agent 返回 GraphOutput，用 .value 获取状态字典
    return result["messages"][-1].content


@tool("write", description="根据调研摘要撰写精炼的说明文章。")
def call_writer_agent(brief: str) -> str:
    """将写作任务委托给写作子 Agent。"""
    result = writer_agent.invoke(
        {"messages": [{"role": "user", "content": brief}]},
        config=make_config("写作子Agent", "s06-cn-writer"),
    )
    return result["messages"][-1].content


print("✓ 子 Agent 工具注册完成")

✓ 子 Agent 工具注册完成


### 步骤 1c — 创建监督者并运行

In [4]:
supervisor = create_agent(
    model=create_llm(),
    tools=[call_research_agent, call_writer_agent],
    system_prompt=(
        "你是内容协调员。请先使用 'research' 工具收集关于某个主题的事实，"
        "然后使用 'write' 工具将这些事实整理成精炼的说明文章。请用中文回复。"
    ),
)

print("运行监督者（每个 Agent 一个工具）...\n" + "=" * 50)

for event in supervisor.stream(
    {"messages": [{"role": "user", "content": "请解释什么是 LangChain Agent。"}]},
    config=make_config("Tool-Per-Agent 监督者", "s06-cn-p1"),
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

运行监督者（每个 Agent 一个工具）...
================================ Human Message =================================

请解释什么是 LangChain Agent。
================================== Ai Message ==================================
Tool Calls:
  research (call_525f9da73a4f4926bbcb55)
 Call ID: call_525f9da73a4f4926bbcb55
  Args:
    query: LangChain Agent 定义与核心机制
================================= Tool Message =================================
Name: research

LangChain Agent 是一种允许大语言模型（LLM）动态调用工具（如API、数据库、计算器等）并基于推理决定下一步行动的组件。其核心机制包括：

- **Agent（代理）**：作为协调者，接收用户输入，结合提示词（Prompt）和当前状态，调用 LLM 生成“思考-行动-观察”循环中的决策。
- **Tool（工具）**：可被 Agent 调用的外部功能模块（如搜索、Python REPL、HTTP 请求等），需明确定义输入输出。
- **Chain of Thought（思维链）**：Agent 通常采用 ReAct（Reason + Act）范式，即先推理（Reason）要做什么，再选择工具并执行（Act），然后观察结果（Observe），循环直至得出最终答案。
- **Memory（记忆）**：支持短期（如对话历史）或长期记忆（如向量数据库），增强上下文连贯性。
- **Orchestration（编排）**：通过 AgentExecutor 等组件管理执行流程、错误处理与终止条件。

简言之，LangChain Agent 实现了 LLM 从“被动响应”到“主动规划与执行”的跃迁。
================================== Ai Message ===

---

## 第二部分 · 单一调度工具（Single Dispatch Tool）

当子 Agent 数量很多时，为每个 Agent 维护一个工具会变得难以管理。
**调度模式**使用单一的 `task()` 工具，包含两个参数：
- `agent_name` — 调用哪个 Agent
- `description` — 要做什么

```
监督者.task(agent_name="research", description="...")
监督者.task(agent_name="writer",   description="...")
监督者.task(agent_name="reviewer", description="...")
```

添加新 Agent 只需更新注册表——无需新建工具。

### 步骤 2a — 构建 Agent 注册表和调度工具

In [5]:
# ── 审核子 Agent 的工具 ────────────────────────────────────────────────────────
@tool
def check_clarity(text: str) -> str:
    """检查文本清晰度：标记超过 25 词的长句。"""
    sentences = [s.strip() for s in text.replace('!','?').replace('.','?').split('?') if s.strip()]
    long_ones = [s for s in sentences if len(s.split()) > 25]
    if long_ones:
        return f"发现 {len(long_ones)} 个过长的句子，建议拆分。"
    return "所有句子都很简洁。✓"

reviewer_agent = create_agent(
    model=create_llm(),
    tools=[check_clarity],
    system_prompt=(
        "你是一名编辑。审查给定文本的清晰度和简洁性。"
        "使用 check_clarity 工具发现问题，然后提供可操作的修改建议。请用中文回复。"
    ),
)

# ── 中央注册表：agent_name → agent 实例 ──────────────────────────────────────
AGENT_REGISTRY = {
    "research": research_agent,
    "writer":   writer_agent,
    "reviewer": reviewer_agent,
}

@tool
def task(agent_name: str, description: str) -> str:
    """启动一个子 Agent 执行特定任务。

    可用的 Agent：
    - research：收集事实并总结主题
    - writer：  根据摘要撰写精炼的说明文章
    - reviewer：审查文本的清晰度并提出改进建议
    """
    if agent_name not in AGENT_REGISTRY:
        return f"未知 Agent '{agent_name}'。可用的有：{list(AGENT_REGISTRY)}"
    agent = AGENT_REGISTRY[agent_name]
    result = agent.invoke(
        {"messages": [{"role": "user", "content": description}]},
        config=make_config(f"task-{agent_name}", f"s06-cn-dispatch-{agent_name}"),
    )
    return result["messages"][-1].content

print("✓ 注册表和调度工具准备完成")

✓ 注册表和调度工具准备完成


### 步骤 2b — 使用单一调度的监督者

In [ ]:
dispatch_supervisor = create_agent(
    model=create_llm(),
    tools=[task],
    system_prompt=(
        "你是内容流水线协调员。使用 'task' 工具委托工作："
        "先调研主题，然后撰写说明，最后审核内容。"
        "可用 Agent：research（调研）、writer（写作）、reviewer（审核）。请用中文回复。"
    ),
)

print("运行监督者（单一调度）...\n" + "=" * 50)

for event in dispatch_supervisor.stream(
    {"messages": [{"role": "user", "content": "写一篇关于 Python 的简短说明并审核它。"}]},
    config=make_config("单一调度监督者", "s06-cn-p2"),
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

运行监督者（单一调度）...
================================ Human Message =================================

写一篇关于 Python 的简短说明并审核它。
================================== Ai Message ==================================
Tool Calls:
  task (call_42209b0d64b14c95ae7004)
 Call ID: call_42209b0d64b14c95ae7004
  Args:
    agent_name: research
    description: 调研 Python 编程语言的核心特性、历史背景和主要用途
  task (call_b220b8657e0e4e5b9b5469)
 Call ID: call_b220b8657e0e4e5b9b5469
  Args:
    agent_name: writer
    description: 根据调研结果撰写一篇关于 Python 的简短说明（约200字），突出其易学性、广泛应用及关键特性


---

## 第三部分 · 使用 Enum 实现类型安全的 Agent 发现

第二部分中的字符串类型 `agent_name` 存在脆弱性——拼写错误会导致静默失败。
使用 **`Enum` 约束**可以让 schema 自文档化，并强制 LLM 只能从固定集合中选择合法名称。

```python
class AgentName(str, Enum):
    RESEARCH = "research"
    WRITER   = "writer"
    REVIEWER = "reviewer"
```

LLM 在工具的 JSON Schema 中看到枚举值，并被约束为只能选择其中之一。

In [ ]:
# Enum 将监督者限制为只能使用有效的 Agent 名称
class AgentName(str, Enum):
    RESEARCH = "research"
    WRITER   = "writer"
    REVIEWER = "reviewer"

@tool
def typed_task(agent_name: AgentName, description: str) -> str:
    """启动一个类型安全的子 Agent。agent_name 必须是 AgentName 的有效枚举值。"""
    agent = AGENT_REGISTRY[agent_name.value]
    result = agent.invoke(
        {"messages": [{"role": "user", "content": description}]},
        config=make_config(f"typed-{agent_name.value}", f"s06-cn-enum-{agent_name.value}"),
    )
    return result["messages"][-1].content

# 查看生成的工具 Schema——LLM 看到的就是这个
import json
schema = typed_task.get_input_schema().model_json_schema()
print("工具输入 Schema（LLM 看到的内容）：")
print(json.dumps(schema, indent=2, ensure_ascii=False))

In [ ]:
enum_supervisor = create_agent(
    model=create_llm(),
    tools=[typed_task],
    system_prompt="协调子 Agent 对任意主题进行调研和写作。请用中文回复。",
)

print("运行监督者（Enum 类型约束调度）...\n" + "=" * 50)

for event in enum_supervisor.stream(
    {"messages": [{"role": "user", "content": "调研并撰写关于 LangChain 的内容。"}]},
    config=make_config("Enum 类型约束监督者", "s06-cn-p3"),
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

---

## 第四部分 · 上下文工程（Context Engineering）

默认情况下，子 Agent 工具只接收你传入的字符串参数。
有时需要注入更丰富的上下文——监督者的消息历史、共享状态值——
或者将结构化数据写回监督者的状态。

### 使用 `Command` 控制子 Agent 的输出

默认工具返回 `str`。使用 `Command` 可以直接写入监督者的状态图——
当子 Agent 产生监督者需要使用的结构化数据时非常有用。

```
监督者状态
  ├── messages: [...]
  └── research_notes: ""   ← 子 Agent 通过 Command 写入这里
```

In [ ]:
from langchain.agents import AgentState
from langgraph.types import Command

# 扩展 AgentState 以包含自定义字段
class SupervisorState(AgentState):
    research_notes: str   # 子 Agent 将填充此字段

@tool
def research_with_state(
    query: str,
    tool_call_id: Annotated[str, InjectedToolCallId],
) -> Command:
    """调研并将发现写入监督者状态的 research_notes 字段。"""
    result = research_agent.invoke(
        {"messages": [{"role": "user", "content": query}]},
        config=make_config("带状态的调研", "s06-cn-state"),
    )
    findings = result["messages"][-1].content

    # Command.update 写入监督者的状态，而不仅仅是返回字符串
    return Command(update={
        "research_notes": findings,          # 填充自定义字段
        "messages": [ToolMessage(
            content=findings,
            tool_call_id=tool_call_id,
        )],
    })

state_supervisor = create_agent(
    model=create_llm(),
    tools=[research_with_state],
    state_schema=SupervisorState,
    system_prompt=(
        "使用 research_with_state 收集事实。"
        "调研完成后，总结所发现的内容。请用中文回复。"
    ),
)

print("运行监督者（Command 输出）...\n" + "=" * 50)

final = state_supervisor.invoke(
    {"messages": [{"role": "user", "content": "调研 Python 是什么。"}],
     "research_notes": ""},
    config=make_config("Command 输出监督者", "s06-cn-p4"),
)

print("\n最终回答：")
final["messages"][-1].pretty_print()

print("\n状态中的 research_notes 字段：")
print(final.get("research_notes", "（未设置）"))

---

## 总结

| 模式 | API | 最适合 |
|------|-----|--------|
| **每个 Agent 一个工具** | 每个子 Agent 一个 `@tool` | ≤5 个 Agent，每个需要定制化输入/输出 |
| **单一调度工具** | 一个 `task(agent_name, description)` 工具 | Agent 数量多，由不同团队开发 |
| **Enum 约束** | `agent_name: AgentName` | 防止拼写错误，Schema 自文档化 |
| **Command 输出** | 工具中返回 `Command(update={...})` | 子 Agent 需要将结构化数据写入监督者状态 |

### 核心要点

1. **子 Agent 就是工具** — 监督者像调用普通工具一样调用它们
2. **默认上下文隔离** — 每次 `subagent.invoke()` 都从空白对话开始；只有明确需要续传时才设置 `checkpointer=True`
3. **只返回监督者需要的内容** — 提取 `result["messages"][-1].content`，保持监督者上下文精简
4. **Enum 优于字符串** — 防止 LLM 产生幻觉 Agent 名称，并在 Schema 中文档化注册表
5. **`Command` 用于状态写入** — 当子 Agent 产生需要存入监督者状态（而非消息历史）的结构化数据时使用

In [ ]:
print(f"追踪记录请访问: {get_langfuse_host()}")